In [ ]:
# MovieLens: regular + cold-start evaluation
# Cold-start rule:
# hold out all users with fewer than 5 interactions in the training split

import random
import math
import numpy as np
import pandas as pd
from collections import defaultdict
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# File paths
# ============================================================
RATINGS_PATH = "ratings.csv"
USERS_PATH = "users.csv"
MOVIES_PATH = "movies.csv"

# ============================================================
# Global hyperparameters and experiment settings
# ============================================================
SEED = 42
BATCH_SIZE = 2048
EMBED_DIM = 128
HIDDEN_DIM = 128
DROPOUT = 0.2
LR = 1e-3
WEIGHT_DECAY = 1e-6
NUM_EPOCHS = 20
PATIENCE = 3
NUM_NEG_TRAIN = 8
NUM_NEG_EVAL = 99
NEG_FROM_EXPLICIT_PROB = 0.3
POS_THRESHOLD = 4
TOPK = 10
COLD_START_THRESHOLD = 5  # hold out users with < 5 train interactions

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# 1. Load data
# ============================================================
ratings = pd.read_csv(RATINGS_PATH)
users = pd.read_csv(USERS_PATH)
movies = pd.read_csv(MOVIES_PATH)

print("ratings:", ratings.shape)
print("users:", users.shape)
print("movies:", movies.shape)

required_rating_cols = {"userId", "movieId", "rating", "timestamp"}
required_user_cols = {"userId", "gender", "age", "occupation"}
required_movie_cols = {"movieId", "genres"}

assert required_rating_cols.issubset(ratings.columns), "ratings.csv missing necessary columns"
assert required_user_cols.issubset(users.columns), "users.csv missing necessary columns"
assert required_movie_cols.issubset(movies.columns), "movies.csv missing necessary columns"

ratings = ratings.sort_values(["userId", "timestamp"]).reset_index(drop=True)


def is_positive(rating):
    return 1 if rating >= POS_THRESHOLD else 0


# ============================================================
# 2. Per-user chronological train/val/test split
# ------------------------------------------------------------
# First do the normal chronological split.
# Then identify cold-start users by:
#   train_interaction_count < COLD_START_THRESHOLD
#
# These users are held out from model fitting entirely.
# Their original train interactions are preserved only as user history
# for candidate filtering during evaluation.
# ============================================================
train_parts, val_parts, test_parts = [], [], []

for user_id, user_df in ratings.groupby("userId"):
    user_df = user_df.sort_values("timestamp")
    n = len(user_df)

    if n < 3:
        continue

    train_end = int(n * 0.8)
    val_end = int(n * 0.9)

    train_end = max(train_end, 1)
    if val_end <= train_end:
        val_end = train_end + 1
    if val_end >= n:
        val_end = n - 1

    train_parts.append(user_df.iloc[:train_end])
    val_parts.append(user_df.iloc[train_end:val_end])
    test_parts.append(user_df.iloc[val_end:])

train_df_full = pd.concat(train_parts).reset_index(drop=True)
val_df_full = pd.concat(val_parts).reset_index(drop=True)
test_df_full = pd.concat(test_parts).reset_index(drop=True)

for df in [train_df_full, val_df_full, test_df_full]:
    df["label"] = df["rating"].apply(is_positive)

print("train_full:", train_df_full.shape)
print("val_full:", val_df_full.shape)
print("test_full:", test_df_full.shape)

# Count per-user interactions in the original training split
train_counts = train_df_full.groupby("userId").size().to_dict()

cold_user_ids = {
    user_id for user_id, cnt in train_counts.items()
    if cnt < COLD_START_THRESHOLD
}
regular_user_ids = set(train_counts.keys()) - cold_user_ids

print("\nregular users:", len(regular_user_ids))
print("cold-start users:", len(cold_user_ids))

# Hold out cold users from model fitting
train_df = train_df_full[train_df_full["userId"].isin(regular_user_ids)].copy().reset_index(drop=True)

# Keep regular / cold val-test separately for reporting
val_df_regular = val_df_full[val_df_full["userId"].isin(regular_user_ids)].copy().reset_index(drop=True)
test_df_regular = test_df_full[test_df_full["userId"].isin(regular_user_ids)].copy().reset_index(drop=True)

val_df_cold = val_df_full[val_df_full["userId"].isin(cold_user_ids)].copy().reset_index(drop=True)
test_df_cold = test_df_full[test_df_full["userId"].isin(cold_user_ids)].copy().reset_index(drop=True)

# Original train interactions of cold users are kept only as "history"
cold_train_history_df = train_df_full[train_df_full["userId"].isin(cold_user_ids)].copy().reset_index(drop=True)

print("\nAfter holdout:")
print("train_fit:", train_df.shape)
print("val_regular:", val_df_regular.shape)
print("test_regular:", test_df_regular.shape)
print("val_cold:", val_df_cold.shape)
print("test_cold:", test_df_cold.shape)
print("cold_train_history:", cold_train_history_df.shape)

print("\ntrain label counts:")
print(train_df["label"].value_counts(dropna=False))
print("\nval_regular label counts:")
print(val_df_regular["label"].value_counts(dropna=False))
print("\ntest_regular label counts:")
print(test_df_regular["label"].value_counts(dropna=False))
print("\nval_cold label counts:")
print(val_df_cold["label"].value_counts(dropna=False))
print("\ntest_cold label counts:")
print(test_df_cold["label"].value_counts(dropna=False))


# ============================================================
# 3. Build ID encoders
# ------------------------------------------------------------
# Items are built from FITTING TRAIN data only.
# Users are split into:
# - all_user_idx: all evaluable users (regular + cold) for side-feature lookup
# - user_id_emb_idx: embedding index for user ID
#     * regular users in train -> unique embedding index >= 1
#     * cold users -> 0 (shared UNK user embedding)
# ============================================================
all_user_ids = sorted(
    set(train_df_full["userId"].unique())
    | set(val_df_full["userId"].unique())
    | set(test_df_full["userId"].unique())
)
all_user2idx = {u: i for i, u in enumerate(all_user_ids)}
num_all_users = len(all_user2idx)

train_item_ids = sorted(train_df["movieId"].unique())
item2idx = {m: i for i, m in enumerate(train_item_ids)}
num_items = len(item2idx)

# User-ID embedding table:
# 0 -> UNK cold-start user embedding
# 1..N -> regular train users
regular_user_ids_sorted = sorted(train_df["userId"].unique())
train_userid_emb2idx = {u: i + 1 for i, u in enumerate(regular_user_ids_sorted)}
num_user_id_embeddings = len(train_userid_emb2idx) + 1  # +1 for UNK

print("\nnum_all_users:", num_all_users)
print("num_items:", num_items)
print("num_user_id_embeddings:", num_user_id_embeddings)


def encode_df(df, all_user2idx, item2idx):
    df = df.copy()
    df = df[df["userId"].isin(all_user2idx)]
    df = df[df["movieId"].isin(item2idx)]
    df["user_idx"] = df["userId"].map(all_user2idx)
    df["item_idx"] = df["movieId"].map(item2idx)
    return df.reset_index(drop=True)


train_df = encode_df(train_df, all_user2idx, item2idx)
val_df_regular = encode_df(val_df_regular, all_user2idx, item2idx)
test_df_regular = encode_df(test_df_regular, all_user2idx, item2idx)
val_df_cold = encode_df(val_df_cold, all_user2idx, item2idx)
test_df_cold = encode_df(test_df_cold, all_user2idx, item2idx)
cold_train_history_df = encode_df(cold_train_history_df, all_user2idx, item2idx)

train_pos_df = train_df[train_df["label"] == 1].copy().reset_index(drop=True)
train_neg_df = train_df[train_df["label"] == 0].copy().reset_index(drop=True)

val_pos_regular = val_df_regular[val_df_regular["label"] == 1].copy().reset_index(drop=True)
test_pos_regular = test_df_regular[test_df_regular["label"] == 1].copy().reset_index(drop=True)

val_pos_cold = val_df_cold[val_df_cold["label"] == 1].copy().reset_index(drop=True)
test_pos_cold = test_df_cold[test_df_cold["label"] == 1].copy().reset_index(drop=True)

print("\ntrain_pos:", train_pos_df.shape)
print("train_neg:", train_neg_df.shape)
print("val_pos_regular:", val_pos_regular.shape)
print("test_pos_regular:", test_pos_regular.shape)
print("val_pos_cold:", val_pos_cold.shape)
print("test_pos_cold:", test_pos_cold.shape)


# ============================================================
# 4. Encode user side features for ALL evaluable users
# ============================================================
users_all = users[users["userId"].isin(all_user_ids)].copy()
users_all["user_idx"] = users_all["userId"].map(all_user2idx)

gender_values = sorted(users_all["gender"].astype(str).unique())
age_values = sorted(users_all["age"].astype(int).unique())
occ_values = sorted(users_all["occupation"].astype(int).unique())

gender2idx = {g: i for i, g in enumerate(gender_values)}
age2idx = {a: i for i, a in enumerate(age_values)}
occ2idx = {o: i for i, o in enumerate(occ_values)}

num_genders = len(gender2idx)
num_ages = len(age2idx)
num_occs = len(occ2idx)

user_gender_idx = np.zeros(num_all_users, dtype=np.int64)
user_age_idx = np.zeros(num_all_users, dtype=np.int64)
user_occ_idx = np.zeros(num_all_users, dtype=np.int64)

# This buffer maps all_user_idx -> user_id_embedding_idx
# regular users get unique ID embeddings; cold users get shared UNK(0)
user_id_input_idx = np.zeros(num_all_users, dtype=np.int64)

for row in users_all.itertuples():
    u = row.user_idx
    raw_user_id = row.userId

    user_gender_idx[u] = gender2idx[str(row.gender)]
    user_age_idx[u] = age2idx[int(row.age)]
    user_occ_idx[u] = occ2idx[int(row.occupation)]
    user_id_input_idx[u] = train_userid_emb2idx.get(raw_user_id, 0)


# ============================================================
# 5. Encode item side features from TRAIN items only
# ============================================================
movies_train = movies[movies["movieId"].isin(train_item_ids)].copy()
movies_train["item_idx"] = movies_train["movieId"].map(item2idx)

all_genres = set()
for g in movies_train["genres"].fillna("(no genres listed)").astype(str):
    for token in g.split("|"):
        all_genres.add(token)

genre_vocab = sorted(all_genres)
genre2idx = {g: i for i, g in enumerate(genre_vocab)}
num_genres = len(genre2idx)

item_genre_matrix = np.zeros((num_items, num_genres), dtype=np.float32)

for row in movies_train.itertuples():
    item_idx = row.item_idx
    genres_str = str(row.genres)
    for g in genres_str.split("|"):
        item_genre_matrix[item_idx, genre2idx[g]] = 1.0


# ============================================================
# 6. Build interaction lookup structures
# ============================================================
train_user_seen_items = defaultdict(set)
for row in train_df.itertuples():
    train_user_seen_items[row.user_idx].add(row.item_idx)

train_user_pos_items = defaultdict(set)
for row in train_pos_df.itertuples():
    train_user_pos_items[row.user_idx].add(row.item_idx)

train_user_explicit_neg_items = defaultdict(list)
for row in train_neg_df.itertuples():
    train_user_explicit_neg_items[row.user_idx].append(row.item_idx)


def build_leave_one_out_positive(pos_df):
    user_one_pos = {}
    for row in pos_df.itertuples():
        user_one_pos[row.user_idx] = row.item_idx
    return user_one_pos


# Regular users: seen items come from fit-train and then val
val_seen_regular = defaultdict(set)
test_seen_regular = defaultdict(set)

for row in train_df.itertuples():
    val_seen_regular[row.user_idx].add(row.item_idx)
    test_seen_regular[row.user_idx].add(row.item_idx)

for row in val_df_regular.itertuples():
    test_seen_regular[row.user_idx].add(row.item_idx)

val_user_one_pos_regular = build_leave_one_out_positive(val_pos_regular)
test_user_one_pos_regular = build_leave_one_out_positive(test_pos_regular)

val_eval_users_regular = list(val_user_one_pos_regular.keys())
test_eval_users_regular = list(test_user_one_pos_regular.keys())

# Cold users: use their held-out train interactions as prior history only
val_seen_cold = defaultdict(set)
test_seen_cold = defaultdict(set)

for row in cold_train_history_df.itertuples():
    val_seen_cold[row.user_idx].add(row.item_idx)
    test_seen_cold[row.user_idx].add(row.item_idx)

for row in val_df_cold.itertuples():
    test_seen_cold[row.user_idx].add(row.item_idx)

val_user_one_pos_cold = build_leave_one_out_positive(val_pos_cold)
test_user_one_pos_cold = build_leave_one_out_positive(test_pos_cold)

val_eval_users_cold = list(val_user_one_pos_cold.keys())
test_eval_users_cold = list(test_user_one_pos_cold.keys())

print("\nval_eval_users_regular:", len(val_eval_users_regular))
print("test_eval_users_regular:", len(test_eval_users_regular))
print("val_eval_users_cold:", len(val_eval_users_cold))
print("test_eval_users_cold:", len(test_eval_users_cold))


# ============================================================
# 7. Training dataset with dynamic negative sampling
# ============================================================
class PointwiseRecDataset(Dataset):
    def __init__(
        self,
        pos_df,
        user_pos_items,
        user_explicit_neg_items,
        num_items,
        num_neg=8,
        neg_from_explicit_prob=0.3
    ):
        self.users = pos_df["user_idx"].to_numpy()
        self.pos_items = pos_df["item_idx"].to_numpy()
        self.user_pos_items = user_pos_items
        self.user_explicit_neg_items = user_explicit_neg_items
        self.num_items = num_items
        self.num_neg = num_neg
        self.neg_from_explicit_prob = neg_from_explicit_prob

    def __len__(self):
        return len(self.users)

    def sample_one_negative(self, user):
        explicit_negs = self.user_explicit_neg_items[user]

        if len(explicit_negs) > 0 and random.random() < self.neg_from_explicit_prob:
            return random.choice(explicit_negs)

        while True:
            neg_item = random.randint(0, self.num_items - 1)
            if neg_item not in self.user_pos_items[user]:
                return neg_item

    def __getitem__(self, idx):
        user = int(self.users[idx])
        pos_item = int(self.pos_items[idx])

        neg_items = set()
        while len(neg_items) < self.num_neg:
            neg_item = self.sample_one_negative(user)
            if neg_item != pos_item:
                neg_items.add(neg_item)

        neg_items = list(neg_items)
        items = [pos_item] + neg_items
        labels = [1.0] + [0.0] * self.num_neg

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(items, dtype=torch.long),
            torch.tensor(labels, dtype=torch.float)
        )


train_dataset = PointwiseRecDataset(
    pos_df=train_pos_df,
    user_pos_items=train_user_pos_items,
    user_explicit_neg_items=train_user_explicit_neg_items,
    num_items=num_items,
    num_neg=NUM_NEG_TRAIN,
    neg_from_explicit_prob=NEG_FROM_EXPLICIT_PROB
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)


# ============================================================
# 8. Two-tower recommendation model with side features
# ------------------------------------------------------------
# Key change for cold-start:
# user_idx is the "all user index" for feature lookup
# and is mapped internally to:
#   user_id_embedding_idx = unique regular user embedding OR shared UNK(0)
# ============================================================
class TwoTowerSideFeatureModel(nn.Module):
    def __init__(
        self,
        num_user_id_embeddings,
        num_items,
        num_genders,
        num_ages,
        num_occs,
        num_genres,
        user_id_input_idx,
        user_gender_idx,
        user_age_idx,
        user_occ_idx,
        item_genre_matrix,
        embed_dim=128,
        hidden_dim=128,
        dropout=0.2
    ):
        super().__init__()

        self.user_id_emb = nn.Embedding(num_user_id_embeddings, embed_dim)
        self.gender_emb = nn.Embedding(num_genders, 8)
        self.age_emb = nn.Embedding(num_ages, 8)
        self.occ_emb = nn.Embedding(num_occs, 8)

        user_input_dim = embed_dim + 8 + 8 + 8
        self.user_mlp = nn.Sequential(
            nn.Linear(user_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )

        self.item_id_emb = nn.Embedding(num_items, embed_dim)
        self.genre_linear = nn.Linear(num_genres, 16)

        item_input_dim = embed_dim + 16
        self.item_mlp = nn.Sequential(
            nn.Linear(item_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )

        self.register_buffer("user_id_input_idx", torch.tensor(user_id_input_idx, dtype=torch.long))
        self.register_buffer("user_gender_idx", torch.tensor(user_gender_idx, dtype=torch.long))
        self.register_buffer("user_age_idx", torch.tensor(user_age_idx, dtype=torch.long))
        self.register_buffer("user_occ_idx", torch.tensor(user_occ_idx, dtype=torch.long))
        self.register_buffer("item_genre_matrix", torch.tensor(item_genre_matrix, dtype=torch.float))

        self._init_weights()

    def _init_weights(self):
        for emb in [self.user_id_emb, self.gender_emb, self.age_emb, self.occ_emb, self.item_id_emb]:
            nn.init.normal_(emb.weight, std=0.01)

        for module in list(self.user_mlp) + list(self.item_mlp):
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

        nn.init.xavier_uniform_(self.genre_linear.weight)
        nn.init.zeros_(self.genre_linear.bias)

    def encode_user(self, user_idx):
        user_id_vec = self.user_id_emb(self.user_id_input_idx[user_idx])
        gender_vec = self.gender_emb(self.user_gender_idx[user_idx])
        age_vec = self.age_emb(self.user_age_idx[user_idx])
        occ_vec = self.occ_emb(self.user_occ_idx[user_idx])

        user_input = torch.cat([user_id_vec, gender_vec, age_vec, occ_vec], dim=-1)
        user_vec = self.user_mlp(user_input)
        return user_vec

    def encode_item(self, item_idx):
        item_id_vec = self.item_id_emb(item_idx)

        genre_feats = self.item_genre_matrix[item_idx]
        genre_count = genre_feats.sum(dim=-1, keepdim=True).clamp(min=1.0)
        genre_vec = self.genre_linear(genre_feats / genre_count)

        item_input = torch.cat([item_id_vec, genre_vec], dim=-1)
        item_vec = self.item_mlp(item_input)
        return item_vec

    def forward(self, user_idx, item_idx):
        user_vec = self.encode_user(user_idx)
        item_vec = self.encode_item(item_idx)
        scores = (user_vec * item_vec).sum(dim=-1)
        return scores


model = TwoTowerSideFeatureModel(
    num_user_id_embeddings=num_user_id_embeddings,
    num_items=num_items,
    num_genders=num_genders,
    num_ages=num_ages,
    num_occs=num_occs,
    num_genres=num_genres,
    user_id_input_idx=user_id_input_idx,
    user_gender_idx=user_gender_idx,
    user_age_idx=user_age_idx,
    user_occ_idx=user_occ_idx,
    item_genre_matrix=item_genre_matrix,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(DEVICE)


# ============================================================
# 9. Loss and optimizer
# ============================================================
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def train_one_epoch_pointwise(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for user, items, labels in dataloader:
        user = user.to(device)
        items = items.to(device)
        labels = labels.to(device)

        batch_size, num_candidates = items.shape

        user_flat = user.unsqueeze(1).expand(-1, num_candidates).reshape(-1)
        item_flat = items.reshape(-1)
        label_flat = labels.reshape(-1)

        optimizer.zero_grad()
        logits = model(user_flat, item_flat)
        loss = criterion(logits, label_flat)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * label_flat.size(0)
        total_examples += label_flat.size(0)

    return total_loss / total_examples


def evaluate_1vs99(
    model,
    eval_users,
    user_one_pos,
    seen_user_items,
    num_items,
    k=10,
    num_neg=99,
    device="cpu",
    seed=42
):
    model.eval()
    rng = np.random.default_rng(seed)

    hr_list = []
    ndcg_list = []
    map_list = []

    with torch.no_grad():
        for user in eval_users:
            pos_item = user_one_pos[user]
            seen_items = set(seen_user_items[user])

            if pos_item in seen_items:
                seen_items.remove(pos_item)

            # If too many items are already seen, sample as many as possible
            available_neg_count = num_items - len(seen_items) - 1
            if available_neg_count <= 0:
                continue

            target_num_neg = min(num_neg, available_neg_count)

            neg_items = set()
            while len(neg_items) < target_num_neg:
                neg = int(rng.integers(0, num_items))
                if neg != pos_item and neg not in seen_items:
                    neg_items.add(neg)

            candidates = [pos_item] + list(neg_items)

            user_tensor = torch.full((len(candidates),), user, dtype=torch.long, device=device)
            item_tensor = torch.tensor(candidates, dtype=torch.long, device=device)

            scores = model(user_tensor, item_tensor).cpu().numpy()

            ranked_idx = np.argsort(-scores)
            ranked_items = [candidates[i] for i in ranked_idx]

            hr = 1.0 if pos_item in ranked_items[:k] else 0.0

            if pos_item in ranked_items[:k]:
                rank = ranked_items.index(pos_item)
                ndcg = 1.0 / math.log2(rank + 2)
                ap = 1.0 / (rank + 1)
            else:
                ndcg = 0.0
                ap = 0.0

            hr_list.append(hr)
            ndcg_list.append(ndcg)
            map_list.append(ap)

    if len(hr_list) == 0:
        return 0.0, 0.0, 0.0, 0

    return float(np.mean(hr_list)), float(np.mean(ndcg_list)), float(np.mean(map_list)), len(hr_list)


# ============================================================
# 10. Training loop with early stopping
# ------------------------------------------------------------
# Early stop is still based on REGULAR validation users,
# because cold users are held out from model fitting.
# ============================================================
best_val_ndcg = -1.0
best_state = None
best_epoch = -1
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch_pointwise(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE
    )

    val_hr10_reg, val_ndcg10_reg, val_map10_reg, val_n_reg = evaluate_1vs99(
        model=model,
        eval_users=val_eval_users_regular,
        user_one_pos=val_user_one_pos_regular,
        seen_user_items=val_seen_regular,
        num_items=num_items,
        k=TOPK,
        num_neg=NUM_NEG_EVAL,
        device=DEVICE,
        seed=SEED
    )

    val_hr10_cold, val_ndcg10_cold, val_map10_cold, val_n_cold = evaluate_1vs99(
        model=model,
        eval_users=val_eval_users_cold,
        user_one_pos=val_user_one_pos_cold,
        seen_user_items=val_seen_cold,
        num_items=num_items,
        k=TOPK,
        num_neg=NUM_NEG_EVAL,
        device=DEVICE,
        seed=SEED
    )

    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val-Regular HR@{TOPK}: {val_hr10_reg:.4f} | "
        f"Val-Regular NDCG@{TOPK}: {val_ndcg10_reg:.4f} | "
        f"Val-Regular MAP: {val_map10_reg:.4f} | "
        f"Val-Cold HR@{TOPK}: {val_hr10_cold:.4f} | "
        f"Val-Cold NDCG@{TOPK}: {val_ndcg10_cold:.4f} | "
        f"Val-Cold MAP: {val_map10_cold:.4f}"
    )

    if val_ndcg10_reg > best_val_ndcg:
        best_val_ndcg = val_ndcg10_reg
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch + 1
        patience_counter = 0
        print("Saved best model.")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break


if best_state is not None:
    model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best Val-Regular NDCG@{TOPK}: {best_val_ndcg:.4f}")


# ============================================================
# 11. Final test evaluation: regular + cold_start
# ============================================================
test_hr10_reg, test_ndcg10_reg, test_map10_reg, test_n_reg = evaluate_1vs99(
    model=model,
    eval_users=test_eval_users_regular,
    user_one_pos=test_user_one_pos_regular,
    seen_user_items=test_seen_regular,
    num_items=num_items,
    k=TOPK,
    num_neg=NUM_NEG_EVAL,
    device=DEVICE,
    seed=SEED
)

test_hr10_cold, test_ndcg10_cold, test_map10_cold, test_n_cold = evaluate_1vs99(
    model=model,
    eval_users=test_eval_users_cold,
    user_one_pos=test_user_one_pos_cold,
    seen_user_items=test_seen_cold,
    num_items=num_items,
    k=TOPK,
    num_neg=NUM_NEG_EVAL,
    device=DEVICE,
    seed=SEED
)

print("\nFinal Test Results:")
print(f"[regular]    HR@{TOPK}: {test_hr10_reg:.4f} | NDCG@{TOPK}: {test_ndcg10_reg:.4f} | MAP: {test_map10_reg:.4f} | num_eval_users: {test_n_reg}")
print(f"[cold_start] HR@{TOPK}: {test_hr10_cold:.4f} | NDCG@{TOPK}: {test_ndcg10_cold:.4f} | MAP: {test_map10_cold:.4f} | num_eval_users: {test_n_cold}")


# ============================================================
# 12. Output format: same style as your screenshot
# ============================================================
results_df = pd.DataFrame([
    {
        "dataset": "MovieLens",
        "model": "TwoTowerSideFeatureModel",
        "user_group": "regular",
        "HR@10": test_hr10_reg,
        "NDCG@10": test_ndcg10_reg,
        "MAP": test_map10_reg,
        "num_eval_users": test_n_reg,
    },
    {
        "dataset": "MovieLens",
        "model": "TwoTowerSideFeatureModel",
        "user_group": "cold_start",
        "HR@10": test_hr10_cold,
        "NDCG@10": test_ndcg10_cold,
        "MAP": test_map10_cold,
        "num_eval_users": test_n_cold,
    }
])

print("\nResults DataFrame:")
print(results_df)

In [ ]:
#lastfm

import random
import math
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# Data source
# ============================================================
# The Last.fm interaction file contains implicit feedback:
# - userID   : user identifier
# - artistID : item identifier (artist)
# - weight   : interaction strength / listening frequency
#
# Unlike explicit rating datasets, this dataset does not directly say
# whether a user "likes" or "dislikes" an item. We only know that
# a larger weight usually suggests stronger preference or stronger exposure.
DATA_PATH = "user_artists.dat"

# ============================================================
# Training / model hyperparameters
# ============================================================
SEED = 42

# A relatively large batch size improves training throughput
# and is usually fine for embedding-based recommendation models.
BATCH_SIZE = 2048

# Dimensionality of user/item ID embeddings.
EMBED_DIM = 128

# Hidden size in the MLP layers of each tower.
HIDDEN_DIM = 256

# Dropout for regularization.
DROPOUT = 0.1

# Adam learning rate.
LR = 5e-4

# Small L2 regularization.
WEIGHT_DECAY = 1e-6

# Maximum training epochs.
NUM_EPOCHS = 20

# Stop if validation metric does not improve for 3 consecutive epochs.
PATIENCE = 3

# Number of negative samples per positive instance during training.
NUM_NEG_TRAIN = 30

# Number of negative samples during evaluation.
NUM_NEG_EVAL = 99

# Top-K cutoff used in HR@K / NDCG@K / MAP@K.
TOPK = 10

# ------------------------------------------------------------
# Filtering / cold-start settings
# ------------------------------------------------------------
# We only need at least 3 total interactions to build:
# - support/train side
# - validation / test or held-out target
MIN_TOTAL_USER_INTERACTIONS = 3
MIN_ITEM_INTERACTIONS = 5

# Cold-start rule required by the prompt:
# hold out all users with fewer than 5 interactions in the training set
#
# Under leave-last-two-out:
#   train_interactions_per_user = total_interactions - 2
#
# So any user with:
#   total_interactions - 2 < 5
# is treated as a cold-start user and is fully excluded from warm training.
COLD_USER_TRAIN_INTERACTION_THRESHOLD = 5

# Probability of sampling a negative item from the popularity distribution
# instead of uniform random sampling.
POPULAR_NEG_PROB = 0.3

# Use GPU if available, otherwise CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed=42):
    """
    Fix random seeds for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# Load raw interaction data
# ============================================================
df = pd.read_csv(DATA_PATH, sep="\t")

required_cols = {"userID", "artistID", "weight"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Last.fm file Missing necessary columns: {missing_cols}")

print("Raw shape:", df.shape)
print(df.head())


# ============================================================
# Basic filtering for data quality
# ============================================================
# Item-side filtering is still applied globally.
item_counts = df["artistID"].value_counts()
df = df[df["artistID"].isin(item_counts[item_counts >= MIN_ITEM_INTERACTIONS].index)].copy()

# Keep users with enough total interactions to support splitting.
user_counts = df["userID"].value_counts()
df = df[df["userID"].isin(user_counts[user_counts >= MIN_TOTAL_USER_INTERACTIONS].index)].copy()

print("\nAfter basic filtering:", df.shape)
print("Remaining users:", df["userID"].nunique())
print("Remaining items:", df["artistID"].nunique())


# ============================================================
# Build a pseudo time order
# ============================================================
# The Last.fm file here does not provide a real timestamp.
# To still perform deterministic splitting, we create a pseudo order.
df = df.sort_values(["userID", "weight", "artistID"], ascending=[True, True, True]).reset_index(drop=True)
df["pseudo_ts"] = df.groupby("userID").cumcount() + 1

print("\nWith pseudo order:")
print(df.head(10))


# ============================================================
# Warm / cold user split
# ============================================================
# Rule:
# - For each user, candidate training interactions under leave-last-two-out
#   would be n - 2
# - If n - 2 < 5, the user is held out entirely as a cold-start user
# - Otherwise the user is a warm user and participates in normal
#   train / validation / test splitting
#
# Warm users:
#   train = all but last 2
#   val   = second last
#   test  = last
#
# Cold users:
#   support = all but last
#   target  = last
#
# The warm model is trained only on warm users.
warm_train_parts, warm_val_parts, warm_test_parts = [], [], []
cold_support_parts, cold_test_parts = [], []

warm_user_ids = []
cold_user_ids = []

for user_id, user_df in df.groupby("userID"):
    user_df = user_df.sort_values("pseudo_ts")
    n = len(user_df)

    if n < 3:
        continue

    candidate_train_count = n - 2

    if candidate_train_count < COLD_USER_TRAIN_INTERACTION_THRESHOLD:
        # Fully hold out this user from training.
        cold_user_ids.append(user_id)

        # Use all but the last interaction as support.
        cold_support_parts.append(user_df.iloc[:-1].copy())

        # Last interaction is the cold-start target.
        cold_test_parts.append(user_df.iloc[-1:].copy())
    else:
        warm_user_ids.append(user_id)

        warm_train_parts.append(user_df.iloc[:-2].copy())
        warm_val_parts.append(user_df.iloc[-2:-1].copy())
        warm_test_parts.append(user_df.iloc[-1:].copy())

warm_train_df = pd.concat(warm_train_parts).reset_index(drop=True) if warm_train_parts else pd.DataFrame(columns=df.columns)
warm_val_df = pd.concat(warm_val_parts).reset_index(drop=True) if warm_val_parts else pd.DataFrame(columns=df.columns)
warm_test_df = pd.concat(warm_test_parts).reset_index(drop=True) if warm_test_parts else pd.DataFrame(columns=df.columns)

cold_support_df = pd.concat(cold_support_parts).reset_index(drop=True) if cold_support_parts else pd.DataFrame(columns=df.columns)
cold_test_df = pd.concat(cold_test_parts).reset_index(drop=True) if cold_test_parts else pd.DataFrame(columns=df.columns)

print("\nWarm users:", len(warm_user_ids))
print("Cold users:", len(cold_user_ids))

print("\nWarm train:", warm_train_df.shape)
print("Warm val:", warm_val_df.shape)
print("Warm test:", warm_test_df.shape)

print("\nCold support:", cold_support_df.shape)
print("Cold test:", cold_test_df.shape)

# Implicit positive labels.
for part in [warm_train_df, warm_val_df, warm_test_df, cold_support_df, cold_test_df]:
    if len(part) > 0:
        part["label"] = 1


# ============================================================
# Build index mappings from WARM training data only
# ============================================================
# We only use warm training IDs to define embedding tables.
train_user_ids = sorted(warm_train_df["userID"].unique())
train_item_ids = sorted(warm_train_df["artistID"].unique())

user2idx = {u: i for i, u in enumerate(train_user_ids)}
item2idx = {m: i for i, m in enumerate(train_item_ids)}

num_users = len(user2idx)
num_items = len(item2idx)

print("\nnum_users (warm train only):", num_users)
print("num_items (warm train only):", num_items)

if num_users == 0 or num_items == 0:
    raise ValueError("Warm training set is empty after cold-start holdout. Please check your thresholds.")


def encode_warm_df(df, user2idx, item2idx):
    """
    Encode warm split rows.
    Keep only rows whose users/items are present in warm training vocab.
    """
    df = df.copy()
    df = df[df["userID"].isin(user2idx)]
    df = df[df["artistID"].isin(item2idx)]
    df["user_idx"] = df["userID"].map(user2idx)
    df["item_idx"] = df["artistID"].map(item2idx)
    return df.reset_index(drop=True)


def encode_cold_df(df, item2idx):
    """
    Encode cold split rows.
    Cold users are unseen, so we do NOT create user_idx.
    We only map item IDs using warm-training item vocabulary.
    """
    df = df.copy()
    df = df[df["artistID"].isin(item2idx)]
    df["item_idx"] = df["artistID"].map(item2idx)
    return df.reset_index(drop=True)


warm_train_df = encode_warm_df(warm_train_df, user2idx, item2idx)
warm_val_df = encode_warm_df(warm_val_df, user2idx, item2idx)
warm_test_df = encode_warm_df(warm_test_df, user2idx, item2idx)

cold_support_df = encode_cold_df(cold_support_df, item2idx)
cold_test_df = encode_cold_df(cold_test_df, item2idx)

print("\nEncoded warm train:", warm_train_df.shape)
print("Encoded warm val:", warm_val_df.shape)
print("Encoded warm test:", warm_test_df.shape)

print("\nEncoded cold support:", cold_support_df.shape)
print("Encoded cold test:", cold_test_df.shape)


# ============================================================
# Build user history lookup tables for warm evaluation
# ============================================================
train_user_seen_items = defaultdict(set)
for row in warm_train_df.itertuples():
    train_user_seen_items[row.user_idx].add(row.item_idx)

val_seen_user_items = defaultdict(set)
for row in warm_train_df.itertuples():
    val_seen_user_items[row.user_idx].add(row.item_idx)

test_seen_user_items = defaultdict(set)
for row in warm_train_df.itertuples():
    test_seen_user_items[row.user_idx].add(row.item_idx)
for row in warm_val_df.itertuples():
    test_seen_user_items[row.user_idx].add(row.item_idx)


def build_leave_one_out_positive(df):
    """
    For each warm user, keep the held-out positive item in this split.
    """
    user_one_pos = {}
    for row in df.sort_values(["user_idx", "pseudo_ts"]).itertuples():
        user_one_pos[row.user_idx] = row.item_idx
    return user_one_pos


val_user_one_pos = build_leave_one_out_positive(warm_val_df)
test_user_one_pos = build_leave_one_out_positive(warm_test_df)

val_eval_users = list(val_user_one_pos.keys())
test_eval_users = list(test_user_one_pos.keys())

print("\nWarm val_eval_users:", len(val_eval_users))
print("Warm test_eval_users:", len(test_eval_users))


# ============================================================
# Build cold-start support / target lookup
# ============================================================
# For each cold user:
# - support items: all historical interactions except the last one
# - target item  : the last interaction
#
# Only users whose support items and target item survive item vocab mapping
# can be evaluated.
cold_user_support_items = defaultdict(list)
for row in cold_support_df.sort_values(["userID", "pseudo_ts"]).itertuples():
    cold_user_support_items[row.userID].append(row.item_idx)

cold_user_target_item = {}
for row in cold_test_df.sort_values(["userID", "pseudo_ts"]).itertuples():
    cold_user_target_item[row.userID] = row.item_idx

cold_eval_users = []
for user_id in sorted(cold_user_target_item.keys()):
    support_items = cold_user_support_items.get(user_id, [])
    target_item = cold_user_target_item.get(user_id, None)

    if target_item is None:
        continue
    if len(support_items) == 0:
        continue

    cold_eval_users.append(user_id)

print("\nCold evaluable users:", len(cold_eval_users))


# ============================================================
# Build popularity-based negative sampling distribution
# ============================================================
item_popularity = warm_train_df["item_idx"].value_counts().sort_index()

pop_array = np.ones(num_items, dtype=np.float64)
for item_idx, cnt in item_popularity.items():
    pop_array[item_idx] = float(cnt)

pop_probs = np.power(pop_array, 0.75)
pop_probs = pop_probs / pop_probs.sum()


class PointwiseImplicitDataset(Dataset):
    """
    Pointwise training dataset for implicit recommendation.
    """

    def __init__(
        self,
        pos_df,
        user_seen_items,
        num_items,
        num_neg=20,
        pop_probs=None,
        popular_neg_prob=0.3
    ):
        self.users = pos_df["user_idx"].to_numpy()
        self.pos_items = pos_df["item_idx"].to_numpy()
        self.user_seen_items = user_seen_items
        self.num_items = num_items
        self.num_neg = num_neg
        self.pop_probs = pop_probs
        self.popular_neg_prob = popular_neg_prob

    def __len__(self):
        return len(self.users)

    def sample_one_negative(self, user):
        while True:
            if self.pop_probs is not None and random.random() < self.popular_neg_prob:
                neg_item = int(np.random.choice(self.num_items, p=self.pop_probs))
            else:
                neg_item = random.randint(0, self.num_items - 1)

            if neg_item not in self.user_seen_items[user]:
                return neg_item

    def __getitem__(self, idx):
        user = int(self.users[idx])
        pos_item = int(self.pos_items[idx])

        neg_items = set()
        while len(neg_items) < self.num_neg:
            neg_item = self.sample_one_negative(user)
            if neg_item != pos_item:
                neg_items.add(neg_item)

        neg_items = list(neg_items)

        items = [pos_item] + neg_items
        labels = [1.0] + [0.0] * self.num_neg

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(items, dtype=torch.long),
            torch.tensor(labels, dtype=torch.float)
        )


train_dataset = PointwiseImplicitDataset(
    pos_df=warm_train_df,
    user_seen_items=train_user_seen_items,
    num_items=num_items,
    num_neg=NUM_NEG_TRAIN,
    pop_probs=pop_probs,
    popular_neg_prob=POPULAR_NEG_PROB
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

sample_batch = next(iter(train_loader))
print("\nSample batch shapes:")
print("user:", sample_batch[0].shape)
print("items:", sample_batch[1].shape)
print("labels:", sample_batch[2].shape)


class TwoTowerImplicitModel(nn.Module):
    """
    Basic two-tower model for implicit recommendation.

    Warm case:
    - user tower: user ID embedding -> MLP -> final user vector
    - item tower: item ID embedding -> MLP -> final item vector
    - score: dot product

    Cold-start extension:
    - for unseen users, build a user representation from support items
      by averaging encoded item vectors from the user's interaction history
    """

    def __init__(self, num_users, num_items, embed_dim=128, hidden_dim=256, dropout=0.2):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )

        self.item_mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

        for module in list(self.user_mlp) + list(self.item_mlp):
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def encode_user(self, user_idx):
        user_vec = self.user_mlp(self.user_emb(user_idx))
        return user_vec

    def encode_item(self, item_idx):
        item_vec = self.item_mlp(self.item_emb(item_idx))
        return item_vec

    def encode_cold_user_from_history(self, history_item_idx):
        """
        Build a cold-start user vector from interacted items only.

        history_item_idx: 1D tensor of item indices for the cold user's support set
        """
        history_item_vecs = self.encode_item(history_item_idx)   # [H, D]
        cold_user_vec = history_item_vecs.mean(dim=0)            # [D]
        return cold_user_vec

    def forward(self, user_idx, item_idx):
        user_vec = self.encode_user(user_idx)
        item_vec = self.encode_item(item_idx)
        scores = (user_vec * item_vec).sum(dim=-1)
        return scores

    def score_cold_user(self, history_item_idx, candidate_item_idx):
        """
        Score candidate items for a cold-start user using support history.

        history_item_idx   : [H]
        candidate_item_idx : [C]
        """
        cold_user_vec = self.encode_cold_user_from_history(history_item_idx)   # [D]
        candidate_vecs = self.encode_item(candidate_item_idx)                  # [C, D]
        scores = torch.matmul(candidate_vecs, cold_user_vec)                   # [C]
        return scores


model = TwoTowerImplicitModel(
    num_users=num_users,
    num_items=num_items,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(DEVICE)

print("\nModel:")
print(model)


criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def train_one_epoch_pointwise(model, dataloader, optimizer, criterion, device):
    """
    Train the model for one epoch using pointwise BCE loss.
    """
    model.train()
    total_loss = 0.0
    total_examples = 0

    for user, items, labels in dataloader:
        user = user.to(device)
        items = items.to(device)
        labels = labels.to(device)

        batch_size, num_candidates = items.shape

        user_flat = user.unsqueeze(1).expand(-1, num_candidates).reshape(-1)
        item_flat = items.reshape(-1)
        label_flat = labels.reshape(-1)

        optimizer.zero_grad()

        logits = model(user_flat, item_flat)
        loss = criterion(logits, label_flat)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * label_flat.size(0)
        total_examples += label_flat.size(0)

    return total_loss / total_examples


def evaluate_1vs99(
    model,
    eval_users,
    user_one_pos,
    seen_user_items,
    num_items,
    k=10,
    num_neg=99,
    device="cpu",
    seed=42
):
    """
    Evaluate warm-user ranking quality using the classic 1-vs-99 protocol.
    """
    model.eval()
    rng = np.random.default_rng(seed)

    hr_list = []
    ndcg_list = []
    map_list = []

    with torch.no_grad():
        for user in eval_users:
            pos_item = user_one_pos[user]
            seen_items = set(seen_user_items[user])

            if pos_item in seen_items:
                seen_items.remove(pos_item)

            neg_items = set()
            while len(neg_items) < num_neg:
                neg = int(rng.integers(0, num_items))
                if neg != pos_item and neg not in seen_items:
                    neg_items.add(neg)

            candidates = [pos_item] + list(neg_items)

            user_tensor = torch.full((len(candidates),), user, dtype=torch.long, device=device)
            item_tensor = torch.tensor(candidates, dtype=torch.long, device=device)

            scores = model(user_tensor, item_tensor).cpu().numpy()
            ranked_idx = np.argsort(-scores)
            ranked_items = [candidates[i] for i in ranked_idx]

            hr = 1.0 if pos_item in ranked_items[:k] else 0.0

            if pos_item in ranked_items[:k]:
                rank = ranked_items.index(pos_item)
                ndcg = 1.0 / math.log2(rank + 2)
                ap = 1.0 / (rank + 1)
            else:
                ndcg = 0.0
                ap = 0.0

            hr_list.append(hr)
            ndcg_list.append(ndcg)
            map_list.append(ap)

    if len(hr_list) == 0:
        return 0.0, 0.0, 0.0

    return float(np.mean(hr_list)), float(np.mean(ndcg_list)), float(np.mean(map_list))


def evaluate_cold_start_1vs99(
    model,
    cold_eval_users,
    cold_user_support_items,
    cold_user_target_item,
    num_items,
    k=10,
    num_neg=99,
    device="cpu",
    seed=42
):
    """
    Evaluate cold-start users with a support-set based user representation.

    For each cold user:
    1. build user vector from support items
    2. rank 1 positive target vs 99 random negatives
    3. compute HR@K / NDCG@K / MAP@K
    """
    model.eval()
    rng = np.random.default_rng(seed)

    hr_list = []
    ndcg_list = []
    map_list = []

    with torch.no_grad():
        for user_id in cold_eval_users:
            support_items = list(cold_user_support_items[user_id])
            pos_item = cold_user_target_item[user_id]

            # Exclude support items from negative candidates.
            seen_items = set(support_items)
            if pos_item in seen_items:
                seen_items.remove(pos_item)

            neg_items = set()
            while len(neg_items) < num_neg:
                neg = int(rng.integers(0, num_items))
                if neg != pos_item and neg not in seen_items:
                    neg_items.add(neg)

            candidates = [pos_item] + list(neg_items)

            history_tensor = torch.tensor(support_items, dtype=torch.long, device=device)
            candidate_tensor = torch.tensor(candidates, dtype=torch.long, device=device)

            scores = model.score_cold_user(history_tensor, candidate_tensor).cpu().numpy()
            ranked_idx = np.argsort(-scores)
            ranked_items = [candidates[i] for i in ranked_idx]

            hr = 1.0 if pos_item in ranked_items[:k] else 0.0

            if pos_item in ranked_items[:k]:
                rank = ranked_items.index(pos_item)
                ndcg = 1.0 / math.log2(rank + 2)
                ap = 1.0 / (rank + 1)
            else:
                ndcg = 0.0
                ap = 0.0

            hr_list.append(hr)
            ndcg_list.append(ndcg)
            map_list.append(ap)

    if len(hr_list) == 0:
        return 0.0, 0.0, 0.0

    return float(np.mean(hr_list)), float(np.mean(ndcg_list)), float(np.mean(map_list))


# ============================================================
# Training loop with model selection based on warm validation NDCG
# ============================================================
best_val_ndcg = -1.0
best_state = None
best_epoch = -1
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch_pointwise(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE
    )

    val_hr10, val_ndcg10, val_map10 = evaluate_1vs99(
        model=model,
        eval_users=val_eval_users,
        user_one_pos=val_user_one_pos,
        seen_user_items=val_seen_user_items,
        num_items=num_items,
        k=TOPK,
        num_neg=NUM_NEG_EVAL,
        device=DEVICE,
        seed=SEED
    )

    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Warm Val HR@{TOPK}: {val_hr10:.4f} | "
        f"Warm Val NDCG@{TOPK}: {val_ndcg10:.4f} | "
        f"Warm Val MAP@{TOPK}: {val_map10:.4f}"
    )

    if val_ndcg10 > best_val_ndcg:
        best_val_ndcg = val_ndcg10
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch + 1
        patience_counter = 0
        print("Saved best model.")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break


# Restore the best checkpoint before final evaluation.
if best_state is not None:
    model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best Warm Val NDCG@{TOPK}: {best_val_ndcg:.4f}")


# ============================================================
# Final evaluation
# ============================================================
results = []

# 1) Warm-user test evaluation
warm_test_hr10, warm_test_ndcg10, warm_test_map10 = evaluate_1vs99(
    model=model,
    eval_users=test_eval_users,
    user_one_pos=test_user_one_pos,
    seen_user_items=test_seen_user_items,
    num_items=num_items,
    k=TOPK,
    num_neg=NUM_NEG_EVAL,
    device=DEVICE,
    seed=SEED
)

results.append({
    "dataset": "Last.fm",
    "model": "TwoTower",
    "user_group": "regular",
    f"HR@{TOPK}": warm_test_hr10,
    f"NDCG@{TOPK}": warm_test_ndcg10,
    "MAP": warm_test_map10,
    "num_eval_users": len(test_eval_users)
})

# 2) Cold-start user evaluation
cold_test_hr10, cold_test_ndcg10, cold_test_map10 = evaluate_cold_start_1vs99(
    model=model,
    cold_eval_users=cold_eval_users,
    cold_user_support_items=cold_user_support_items,
    cold_user_target_item=cold_user_target_item,
    num_items=num_items,
    k=TOPK,
    num_neg=NUM_NEG_EVAL,
    device=DEVICE,
    seed=SEED
)

results.append({
    "dataset": "Last.fm",
    "model": "TwoTower",
    "user_group": "cold_start",
    f"HR@{TOPK}": cold_test_hr10,
    f"NDCG@{TOPK}": cold_test_ndcg10,
    "MAP": cold_test_map10,
    "num_eval_users": len(cold_eval_users)
})

results_df = pd.DataFrame(results)

print("\nFinal Results:")
print(results_df)

# 如果你在 notebook 里跑，想显示得更像截图，可以再加一句
# display(results_df)